# SemPointer: Kaggle two-T4 backup evaluation

**Attention fix:** explicitly expand grouped key/value heads and require PyTorch's
memory-efficient SDPA kernel. Both T4s passed the 16,384-token diagnostic at about
832 MiB peak allocation. Full-model pilot execution still needs validation.
Corrected results use `kaggle_results_efficient/`. If the earlier completed
development run is present, its frozen routing choice and raw records are
validated and retained with their original backend provenance; development is
not rerun. Earlier failed pilot records remain in `kaggle_results/`.

Attach the existing private Dataset containing `sempointer_qwen35_frozen.bin`.
Select **GPU T4 x2**, **Internet on**, and file persistence if available.
This notebook includes its launcher; no second code Dataset is needed.

Run all cells with `RUN_MODE = "pilot"` first. This runs the original development
phase and representative pilot, then exports results and stops. Inspect the
projection before changing the mode to `"run"` for the full suite. Pilot responses
are reused. Sources, selected questions, controls, output caps and metrics remain
frozen. Precision is NF4/FP16, separately labeled from the L40S BF16 experiment.

`HOURS` is an allowance **from launcher start**, not from Kaggle session creation.
Keep it below the session time and GPU quota remaining. A new full-run saved
session can use 9 hours; the default pilot allowance is 2 hours. Reserve time for
setup, exporting and saving results before Kaggle's 12-hour limit.

Download `sempointer_kaggle_results.tar.gz` after each invocation. For a new
session, attach the previous notebook's saved output as another Input; setup
automatically restores a single matching archive. If several exist, set
`RESTORE_ARCHIVE` to the chosen archive's absolute input path. Persistence alone
is not a durable backup. A session stop means partial coverage, not completion.

For unattended work use Kaggle **Save Version → Save & Run All** with these clean
cells. An interactive tab closing does not provide a reliable background run.


In [ ]:
RUN_MODE = "pilot"    # "pilot" first; "run" continues the complete evaluation
HOURS = 2.0           # e.g. 9.0 for a fresh full-run session; use less time if needed
RESTORE_ARCHIVE = ""  # optional absolute /kaggle/input/.../sempointer_kaggle_results.tar.gz


In [ ]:
import base64
import hashlib
import json
import os
from pathlib import Path
import runpy
import shutil
import subprocess
import sys
import tarfile
import time
import zlib
import torch

WORK = Path("/kaggle/working/sempointer")
WORK.mkdir(parents=True, exist_ok=True)
ROOT = WORK / "frozen_bundle"
PYTHON = WORK / "venv/bin/python"
RUNNER = WORK / "sempointer_kaggle.py"

assert RUN_MODE in {"pilot", "run"}
assert 0.05 <= HOURS <= 10.5
assert torch.cuda.device_count() == 2, "Select GPU T4 x2 before running."

payload = zlib.decompress(base64.b64decode('eNrFPWlz21hy3/Ur3norBdJDQYePnaXD3dAyPVZGlrQSNZMNo0KBBChiBQIcANQxCv97+ngXLkp2NhVP1YgE3ut39d39mn/8w946z/amUbIXJndi9Vgs0uTNzqtXr8b36e74rfADf1WEmZinmSgWoYiWy3XhT+NQXIbL8zRK8OV0nQRx6IpP4dxfx0VfrKI4LUSaxI/uzs5pKqJkHmZhMgtFtk5yeAFwVmlWuOIqD3MCzDCcXIQPUV5EyY0I7/x47RcwsJ8EYhkWWTTLXZzbzjxLl8Lz5utinYWeJ6FBuyQt/CJKk3xnRz3LblZ+lofcZ5bGcTijFqrTUbrGRfREwLMPolmhGsOLhyKOpqqtfLL0E/8mzNQQ81lSxOrLws8X0IMh8DP46sL0YSsLX0E692e3AOM0LT7D+MEoy1KYwl2Y5TA3Besfufmc5gxy5RcLa0bn8FU1yRfrItITyaObxDfffoujInyjv66nqyydhXmunzzqj4WfzaM41F/D5ar0PVqGOzu/nl38LAY0fsfZg7XcxOHefZrdwtHt5dCFccPp7pxdjc+vxt7p8OsIOjjc1MvCHDY798L5PJpFYVI4O8PxeHQ6Pj479T4Oj34enX7C5vq9lwcr3wsfVjF8L7zbO2fnZPTT8Ojv3snw6vToy+jCu/wyxC7Buz8F8x/fvn/z/nB++H56OJ0dhD/+6e2fptPg/f679wdvDuHZm+m7tz/++fDHtz+Gvv/ng3fv3755++cf3/w5nB44O5d/vxyPvl4CsIljraUnHPnRS+dz/DpdHr7zDvWnt/rT0i9mizDwpuvgJizw8Xwdx55EIOd6Z2dnFvt5LkaM5XDqn/0oBnTufPTzcPQwC1f4sNvfEfAPkH6Uz/xVSLQCiPB7mCAtJTAroKcsiBI/exSrMNsNH/zlCugzRJQSWThLAaseXSSbnR3AcYFn6SGadhCX5AD3UbHgw6SHbroKk46TTZ2u8HORF1noL7kl/stCILxE4bpLEIPoJsyLDjeF9eYL//Dde6frLsIH+a4rZyC/4spDOX4FInfuIAG4wXq5yrlxT+SAgd5t+JgPxhl+D5McWYCfz6Jo8NmP87DrAqNJg7DTbRoaJhfQEjW9D07TpDILGjZO/SDvWFuCXT08PYAsorm9W8Sz8k5XhDABBViOCBxsGc3kmPaS8YmiIAKjn7rAsQDj3eVtEGUd/qIXjEN56S195S5IoACIeuI5evkaiOaBgLr8WfwgHLdYrhzTw73PgCPwgur7HCUBDDo4bNxhhPZfiQ0rC1exPwvlOiSeZWH4e9iw8Oa9M9gVWcfUFX8YcGfznk7Kj2Cnf8EXxDs7c+coTebRzTojYhKzhZ/chAFsv3hCQJsPADQHacIkRPSJUgaICBka0JBYA0RfJOG9SNfFal0AngL5QJdHuVQ8XTONhoOVKweCi+aPHou0TpamhcIv+KiOnB7TUxAn0RzmBm9o3dRqTzjquYvHI6cAewMyznSBaUsMf7rt35GYvu3dwfHpJi6c8hJxE7re4m46sGQvCpxNd9I/3L/GR6rtRL27tmi9utPOZ2Y/eg7A6Ga3+XopllFO++oYVM6Rh+oVIV068CHxl4TKK9jeMOjStMvP7BVMqJ+HXCZ3rtV6rq0xfrAGySYOPnSue/hRcqFrHiMrg83SewAI2GweBeFdGKerJQqkax4CO/Ih25OjgUs4i+dCJAd4lsZ3wIDcKAc5FwNC3gGhpTRD87aLh1dhxngYapRnEV6eQ56us1m4p/h+7TT6kgDkqUgep5YsUTYHsVp4uCEd/F9PoCgPM4m3N1m6XuFRWjpSJwa67eodgl64LdjZTJz7TeDhxJn7yyh+hE11/RUIF0Tz+9KMJqxlcjuEhbw+DDoMxDrA0vMJt4fTBqkwiP3lNPDFQ/9h4iAadye8jn7/8PpaLnWFm+KFyU2UhB3ePLlM5hm4zie9BJD/8dxN5m9Bdi+BKYCgKx5BDg9E7VkHkCcCYIgMstFAvIJmr1guoKBx+i+ADCxntnDnIIGKg/dbofUMtNt7UHXpjPBwZsQMB/ynJ+J05sdMQR6q5VKYENwBDzeV43XtKf5vgWqYPNGNRpdpCP+Hxv4cDYhI8etcUXeJsviQ3Blq6h3uSZRy8AyJOFeJJlmpM62Ao0eoY6MCHYdI5yQvUDrMSQAkAnAI5Ei6zhlVJN3gP54IbIeckRJ7pdWUkJobKsyD9iDMPSlzOiwgekoG9TRRKnHBz1+qBygu2s47zX6Bjg96KUocKf+YKVfef6Miwgz5ETUngCyXV+X7UpGaPhYhnHOJgeLcB6jza51477f7MHnzzl09OuWzNsOUyFk+doNQqYBKGaxyarlArbmhfcuP7OkhlkmY5fFlW9ahuLFsVzr+J1xT3zD58m53X3BkG8Cdf6vYnYhLxVvPLwo4D9glZVZ0jLVwnoEgSwoBJvxPfxsC+ea0T4JsyBxAg8n6KIADoTUMatBvaz9AzWmGmtFCXH46H5LJQHiFfZiek8TVoyoTlOwy4LFJGPew40cwbYG/W10zP8lhpUuwbl00n25YR8tLEDRclusw/whJcFB56wKReje/+V6UkEVIrbc3ARhSLrxGbtYTr18zX+sL0mdZjc0ezQmTNWStq2OW5Y4+fz4+OgZD1dP2areMG49RGMvlg1kWxxbgZyeq1i05BksvL07TFWkPwCIAN2I0XvwgBowHi6hIV94KsIusIaQCbxbntlUD53ixTsQ6UTqx4oXYKf/AGgMZtjlBA+Zc3IfQ4CYE+5KPqowMhkC1ZyRO14HH0+0Jptqm9i4braobzOsSdfIt2nEODDr0pLGkFEl6yHoxNUqz6Ab3WvIBaCkZx9/oz4ielluiZZwhA1FzcPnJjtKzzWaKKCdxawkl6QpBotAPa8iump2kN1GRn7PDJc12dA92AXyOEuAi3KhTaVtBLcQJzwOxG3teB/UHtNRAdwCdHfA6h/mHeaWLXE2HCTjKEz/pyIaunzyilgoiV3JA2WaV5lEyb2xWg63+ISdLCw1iTosqgQii5WD3AD7HsQbXMFkjyj+j9gDcifyMUqTDOewybAs9QRXBXfvA6IvuwijJUfAbee/UZ65ENE2xeijjt4w0nQpqNR4IzkcfiOIuDUvL16swg7XrHrpt05G16Id/qCiI23bQVoZAFCjxs1yD2YbajnKnic/noGnKcRq2iqZCrMc1u+6xIuiytezxPgJBGaZq/5P9SHuqQQt3au1xZ4kzhYGnWnU0A0/JO5a3oI98O3EYL7yVIiew9sBULBGcNCSr/0gEbBOyjZvOGNU+3Z0X7CzuUHXhOzWkA/pMCeEaplOXujRA7hJT9N4pvkRPgb489dzPQRtBa7WNVp5ev1Y4TBPo9oSTwN6A+gCb9DvMmHDU6TslFLXslOo/h4nZkyd1s/azwOmziulMSbfLAbDTP9gCwxzSlKW00685k7dNwQdNFiWM59/5UYzhDafPG4EmfMPrjTkQS0wMNNfg13hOtCDYFxYuHfJA9dBMts6tIpPsRiWnWHoPOFJ0SPqtc6fLyjJSt9NkCdVdyxoCd4IdNm3YYQx2kH8vcD+XqwJP3YfVBE5XYm6DVIVVmy2gRhWBCg3Km8C2tqUwuN7o9KfjUwwPoJg1Yl95xIwO0BNPDuktTn+/J9RO9NFvlQAm45KAw8EnePi0QYe9nxURbC3yPXq26e6U2AspPjZRE/jGkTdd3j1/Rh4dngfPIkM5ZTHOVYZQ587Rl9HRz+dnx6dj8UTuH9T2N+KJ+m6g8zxe5wtpQzUoogtQ1XkvJnLd1+JfLZ2vfO6AJGwwpiCK0ySagVj/y0Ari2JXvN9HQc2eKLbnUJ26HJ+dO02uWM2lSN9arwLkarzpA9haECFA8RK6o0yrZ6Cz+0L1xqU0CJwp7P7tTm0KyBptxEF3Je1Dm3Zsjrid/uv/2FzoP7OQEnFWzgh2ve2QZEuJuiiQHBS8cVjwFtYhvZdbBm9uQk83rnqibVylz91qwIaesiubAkyiFGtC9g/PLWuljQYr0Ldhh+I09Hcwd57IUwbDoBaExOB5m754ggcbCw1esCJkcc1GVp3/VBisbtnMzioan6UYtrMs5eBJ/wFaJwyzBL4JHKGjjPpcLgD+qvMLbBtNMq2tLlY+jkgeo+VmNW5TGYjIdStmWIolMg+rudisCSh3KiB5gU5aBicjPWnhk5SbGMd4ZJxWOGaYrJes+uiFW+MEKe1vR+8A8qIJsumnzXV3EhlNjBMIUFMsOsYvQk/R1HGuLVRZB1G96ew+9OhNtTk6qHEJ2ZaAAMpa9h8TP0NvP80HDYvKC1yS3GOAABuM0+haOiWN98PLB4TXvKKtg+HyZv7qzWHDiHRMsMR9t4QgjQ562yEmEaDsrK+Lgrph8TXKc/QhmWPl7WKINLgjfhAlwGWOz1P+AQNRD53miXTFa3HgviuvMlexBPpW9rw5khhDRbJeDsSfADb0uS8qB6YJmtS6AU1DNlJh4n0Me77Z3y/LESf352HxCDoiBgpBS3Xfkd6aR6iV/JpmGB+r7MtvQFHEb1ehDnf4sywFq3OagtXB08X5lYd6t/8vYgGsGKTSsidWoDgA77sDay5KwGrL+ZRD1Oxh/1xxnMzidQDPYWkijx6Eny1zGlNNwK3o5c7MX+NzmPk57wzOEl37HwSlPWTrFS4DJ4jRVKD1nAwlsADZgY/nzU5FaJZTQCp3nY1kj2BtYvjeB8sA5t1hu9E4qS4Tf5UvUtqwZVQgiMu/naClj5kz6OnABeEQvw5PQD0GsRBiVpKfQI/VI76IcT8+fTTuKhnJlV4lOSK9kbNAfk9PLTlvUj68craKC4qee/M7n8sfxRmwM1SU0wyTLz59xGCzL9iVD0DzKOBgc+7fhcFuNajsigv/XkKSCndOVL/KQuT+5CH1YxS+j7wwIF/ekA/Cv0ujQARrNN9RKvDW5q5J5lBpO+5YzfCTGrkDI8yjh4G10F0GsCvzPWYZRQ4NtelVDtTGlVIN7OQCMwPOI+JcEg0B0Pq+D3vIghA2No8x5jp4QyMX8zLHUcFXK+4njyu7idNpx3ntNPmQVNRVpUXYUVgbCapOMh3DRbMP5s5uMfUkf1yCVnnLDx1yvDnEUyV0RKEi3+YhI0io9bjAtXLcpU7H2QXmg2bEbr5Y4l83Tme39IE2tVvqSDYfKaLbxumY9hHyQgoG5OQwhaEcC8P0M4WC8sFmq7qMaNqxUlWQahSNcKqZrSe3eEzQoQKsq+5sAAzjpWLiEafgVNLGAE339LY3HqWd/IJbpqbVPBUdDqPBJPp3iRdIpqT6tzuLZIqdC8tKgM7A+EMM6nPw/a9omQyyFI51nUVs7xGlUXyw3aPUCJlnS/2Bvxf9rUclQ5XoGFmvOti+2YdbzF0/CCTsnjqDgfzb3bJt7jphsqi1KSfMNIzGqn3rWJpp6EirZD4lUS+f6Qwvcp/oBEMlbeRfaU2gCDOS5+cwXBGn1mEnEITpEuNhuxgKQ8cpyDXAdeS1Mjk0y1jMKbxXIoeko8QkGkY/fWkctc49y5N3MmSgdP7yRdkWAOgZZVAoVg2HtAyXU1Aqqmq/nYRG3dz6aSNjVCzQn+ZpDNqG5IGu66hkGGZ+KgxgnuxsY4XUYsJJSFUSV6DkvDCkoBiyeoK72chetqYC8AC7SgHgnbGj/JTCaEk6sIUy0O9o/Mos96QP3ErR0g4BglILHTVP74LAaaVkgZ7YlNBuZqe1WUPwc5ilzlaUuZemiXSGotOEZ8neq5qztDRv1UnaHB2cfU/U3Knd53MvbpP0Pqnutgkf10fXXJhIaE907NRha2rAz+u5wuwiqYqJnVIIjOFTQoJ1ai1+L8lduJMNp5I68GyeFiji0gJQvMIogVrBY2iUrsjwNx+QG+VhhopfIRNmJHcr+xpfrO5J1rgLQggmMCDm1KjuWfqxQn4M2sl2PcxaA51x4KBeXnHedWoStHokwMuRxygptlPfa5mStl4u/Sz6HQx5GLtkLLQEoTm23Bh2tn03ujnZSs+YCdqPQU4DdITJtnti7tys1k/RZo80wb1Kuia7R0g9xKh751Ca6Oy82QLNim3b3u5NK0Ttr+L01oHcB7cUXDdOLVL0bTcXdXP5ubRHpN3Ffiy+oIFxAZn/lAINZGtg3+KSzwjWE6TEMZIQc24FW9IS2L9fnp2ekImGPUNMR9c2MbrMMNJQtnvchnW5euIymQODX30TMCj5zWs7VXN5NGy8wdB2RzjjTL2zpIKaL7JtDZXj4JBIupxitEWbpMyp+CErntIvqlvuSTOkgnw9UcYthw/Oq6CoSbCgRJuyG5oIvuKGJi+k8UAaVz7FWFqnWMFoFbThv5uSg/OGDJaJ8qZZji7LFaUTi1vPcO+JYOnvSntvIyHLD4aZCisTrCADVCf8VjCiZsHInHd78Vqd2ZPrq4ifWlziSZ6W0wetwA920f8i1gm5YuZEisV9SuQo2Rly4Smm8uCGAe07/crpl6TdRsaZOM1heHOThTdwPMQpLFdtPa9By6yeVCcqm0GRbCPYJC/VD7oNjado4lWMG6cPChm079umUrUvNLQc0S+kekvfbSd+PKbSQtiKrtu3rUZnO/OwFg6twhnq0XMHVJnh0RfxaTgefhxejsRfxfBS3CNP6Yk2I7LbbQOOO6O8onPncnQyOhqL1+LzxdlXAurGPtFW43HoWTlHF6PheCTGo6/n4pfj0a+Cu+HU0JfriKtTUL7E8OQEjIB/pFHSkWN3rRwHRqKQHGleU7KCCkvBm1qvBzulpiGfRk0ZJHuO+Z2l/sxVm4ZcAdbvqOxaW0oCVpSJoS5VKAE5p4AYgVdk3sL76wOUpYEM/BQxhTXrLGPvYnR+djF2l5L5U1P72g4/sO4laTPZ+aP4mgZ+DGphsjsNk9kCxPStpRAhq/ij+JlUM+QneNfTetstqWW8bp0+TzcXNYLzZDFnflPLQp5wGghF5a10FPzuz2aAMGhP0525qMhBBaBEWvyerJerR/wQSanhrB59UKfvCVQKVgDohQioTgbOYn2Dfug57IO3WE/p7l3s72J2kvycL3YxOuxnu9oaIVnjr3M/hobJ3UHpCkwJCcyaJ7hMjN/Ka5udsuksQ6xNNz23QtMajdx9fi13X8cSPOXEz8uqsT4OO0jXzBNlOO5pU1KbXiqEX8JN28RwOZsJpjGhdtcyjFdzPzzDV7/f/RZMm7l0eU5PGIrav7bDaBavbPW+vZLcV97M8aKA2bBkpb9+GV2M8N4tmG7i+FR0/trT/3XFq3awQ7B2pdKG3RxKvAucnpPfRsD4waIWP12cXZ2Lj3+3x/4y/OX49CdxdHZ1Ou58Or4cH5/C5Hj87uD9q56Q1167m50yaip5grvSbUVNedXU4yitjOJJ5DRRYh0IN963oVj4WSBuozgWMwrmYPgHga8AvcG2kLYvqD6RCV5RnCWzEjy0A05OhBTpcqhaB4hVOkE5Zv17tKoH7G33konSP837GJxX8XEOwX9L9H3zPbFuChZGfENJxob7mQmStkeY7RPFDIj2RIMXkTXibytNb6fXfy6tzjnWQRcvet0KYSrtR90VsYjPacs7Buoe2Pd/S5c6GjQ+ClIbIuMAPp3dtixROoJJrW/9apzGXbyN2+NkAgo4EYTmTA6YFSIHE7aNFNybUg108hFFh2oMZLNBhywipGIJ9dU0Rm/mcZpm3pKubayXnYwJwEoZXiIZ7HfLk+o2w5D5ABJIhNfD/pcw2Nng5WtUofg21ounJanYhV2RlNNR5Dih47nuAbfpat1bzWBPHOzv73dLBEjcSbXEnFpkFnbqovywqXBb2bNynUzmC+jrZFWWi/ZawBeQNNvdohbIaxh1F4duoe/zcgi0+y33wyS+59ZlDvZgqBdO1SBXL1oYin4to7MY4jU3zzTUnnQuelHuaduuMjeZG2Tdha3y0Z6ILJdyYDPiMu5sLHMki+5IFsFBmyNRLhmeqifvXffrN7ExXk4H7PSjxhxG+7pY3zpp6KiSsZM5VabQZVi85xOrvz+nWjl8zdWLvnP6+a24D6ObBcrVxksH5AFM0mRXp5LTG9A4AKWa88f5TPp8X9kSmP3JAx3GQ1kSWhIVUYvewrfrlk0tJZC1gKxkmZXA1jAAuC7iIubbs/1IuZ+gSDw5KmXG6RPXtCT6YCDmFa60YWWjPBPFipz6uO4qXXUUOnXrs1KIRrd8qZSAfMX1Aaq+rTLVVv2OqqtkUexp94zVYt2vl3evm/gV8dOoeLQusFEmuvCR6e2S+pxBN8o6W6TRLPwgEnQsK9c+2HawG5lA97WdT1O+xobB3vAeM8NQIOiCQKBfH38ajkeXPSVoe9Kz4hUL0HoWaRxwWLO8Ap0+wHfKe/VggkqQln70FwSiJOs3V2rMdWhdSACRRCXFqGfb6jd8Mmeg1oxu+5z89jLG3VIBQzkwdFJw80KoLVZp4Qo6TZwOKJaStT08hFxa4PJ+kmdxNGR/xDg8WcqhkrJmuEz5vghyWNKACIRyWjjXahPVSlgxiH28HYmYIKvWoP7TVF6IqlE9lm2+EihYNXVWKGweUm2OkOLyuDVbj+gkvPFnjyKwT0oChDOZAxvPy+FB5J5ACSmVMNGnaPeXTmXp8/dZJFUOEQT9vlHyFa578powKMX1SiQEqRlfMfYzTEqZg/aE+J5FlLPXXV1KZXJ2ZZK6jNgDm+xIMdtTFmuTsVOq3sHeDDavyZ1hV3JS9ZokuwSzM4gCFZi7ltdR76UkLTu3dFuEapiF5VexfODNe4te5g7O1kNnqgY4cdAH1JCtLNlQkzN95UdZrjbINia4/Ik0Arpt2gkZawjBKjyCSB6HCRUCIVzGL6ZYiyQB24rA7A1pRFSE1XNZAk2IzhihI4QUGoQ54sQSu/6V1u0akiIIsc1BVGMuaImXUgPsBAnNkbrl2jgVfnWNO2H3Y22LOpHGtdOQ9WK3z0Py1ZTGoGc1yBY/LDGXSRPHvP6eTZdevwqPYTEZFbnEQDgC2oS9EjMpE8ukhs6WZiFTOFllswq2ZP3t6GvFOwypWgaUfki8vlhgcDGzyEWetk5NHtQFescA1gxOtZ8YDe164pg+jPrEAtubbOPzl8SH7YNg/ieCNMxlpScs20BH4N8rRqBL+aiyinbE2ILm6RbOy9kXo8JAtPMnE92Og608jkA11BzR825u8i32pNRKm1N7VBw8DmoqSiV9ps0jxV1flX1dr9rTObPZC9xdZrLVnNn2lE6AXM/hVCq5tZ9Oq9juMaLaihzKDaWeoV5m1D2iH63Iqja8g05faTxV00m1a7AdHZ1RSaUuKI7DBvn3GJoVJEeiUKpb37CiWi8S/iu81SqvOVxII8LQqxIuxudspXqSCemKM6UF24RbZfeOylaLcn2PFo93laV3YeInZLOkZKGwQVNK+8+UOQL2m8zFbjrsqvnFB1NCDsMYXqbd9eiSN2JKr7zNPB91+JttgzSnML2xL+xW7jK++N4uX6x1LsJ15c5RSZqpJX0wFos6DrpXZVipORt0Hjexzj23ck3X0nPxgbn7d4O3GcrhOMxUjcL8RfG40nXLF+eFVTaVLyJvuvWg9f93nO//KI7Xo0AQTG3wbmtIj/fBBPUo7tEQLeDN7HGY7HW3FLLTYTV1A9/ya/JBa/fvq5/Or2CH2MoZPJUv9W7EkxUfUsDgKU9y86qEZI74b5VkIUfR1UyBZ88WURxklCN+o+v25eW6QOo+tKo8yuUwbLTk1GHrgXZpIFhZ4mNLnp0ctnrTWSZSN2CzvEYbpzeOqjaLQmG6RsUTUHhw0G2Apfe3murMR3AXZWlCHIC8rWnuymeA3EdXn4beL8eXxx9PRt6n0S/HR6NLFnMRSrizr+fe6dVXb/zlYjT8BG+cQ2frlXFnfAaS6fg/RxeX3vnwYnhyMjo5vvwKHedo/CI9ffnsfbn66J19/nxyfDqCNwf4dHwxPL38fHbxFXva7zaV3LLl0qfk7QmowRJNsfYE3hRaozRXd2M8Uvw9r2vVlsRGu9JvzE1hlW3LgaZ8QLKp5GAEQxoY8g0hg3zE78kesN/Sg61j6YIB3EsjJ8FDtN1VlRVICdF4XKnUorFY44Quau2eE0bJLUQ4ATIIVZsYvoZZNrDaX44/nV2NsYDW3cBCI4u+Y58KtFqXamliIIRI/dZJG6ZqAycSrlKsdqSuAVjphGr2FT6YUASyWsWhfmH3fpdm9JeBeLPfkOZGUrIqkupirFz0gFYIoKujlVZSvhew331+SRQyMhqCrKJQpOvZor6yesNtVSmsE7C+0MWR+4Y9s4th/HDApTCsftLzQAfF4Yj7Xev1X2Cru80xaFx6aQv6bQFjbFFBivZIMTUGlRUUI8xPeEmZDMKcPA7DFegWO98wx1qSk57AvR9hrhkL2YP9CtPlLCeLlsbccvSwQp9NC0xM9mhYjxmv5J5hBGRxiJ4OtLr3n0c8trcvmF1Ji3uosmbxYFd4mRmVDl+WvxF0hdrH0u2k+lNiisIYVzjP1RBxjvTV5ao6r/TMD7o2GUjC15YctK54rFETdttyyl9wmC9FtGcRrBErXoYZ34cdz2FIG5awSstXrhKlLTSpIyZhVCbUgUqFvwrB4ktiECnSbXy4VTWSQZhvueVXztKWM+ELwEYtIr0Wn1k4gL85geW1Zrcd/F9PPjk5O/rZG/0HaIzW99OPZp+sYuPlMuW0GP5coj3LNynjKeT0kokh31CBV7riYdMxrZzrfybK3Faj2De2mmos4vUrqiw2Wwe+ywWXPS4MTJ7RQ+Wetlqhk0G19Ff+NAID4xH0IWzf+VPvXcPFmmddqOMF0JT6cZQs/G0dYe278MGfYV1VTNAfr1GNpQKs51e2j9poSQqF+AvMgX5UY09GxsiD1u59trfEFJi14O1ZmbLqTlLJ16zfsv6oSqU/t/TzKEFz2STe7WlAyGiWshoIuhq53ijWVuYLtMXaLhKAvzHwoOzd8sS5IB7V1giTPKW6sdC4es1SphSgtl8ptFsCV6+1K/3QmOFEgCcqyLj0V861y2l8eLXU6OQ6FFSt4mwRTk/YCrJVz7keoHj6lgSQanjSWqhW/ps07irS9NvRqRZn7et8jaYgQ7/Zw1YJyVLmiTdfYQXAajU/E5/tS15ix2n7Js28cYzvzU5B8xPTOZrZA91LjJou2jXCQq9AMnvEypeUZnMJDNWPqcQK0PwHruYCn5R1DPjwqO70ztZZRkWYPwiugLjLFRC5Vh4YImjqBZZNiEYg38JO8jArOvsNFlmtynXbJSBzvwn1XIJh+7447GO04YpysS2poxFWc04a3SGyIscvDmZUsk4qxDVp7qT5G8C3CnKr113+2RH5rVKsrmTd1Vye5YhDdV215o138Zp4RTW/r8Ro6hsq8/sMwtDPQbl47/JQs4metbe1MjdYZt30r1tiss6D5PD1WIyuMziQCtQPjFoLWHH++s37/X2xKw4O9+0VUrkFOAtcVVZxU3JWOSV8U53ADv6m1RqwaZ4hRy9j5fMWJqmIBIHcnvwDWe7l8U/Hp2D6m6/j0cXXqgO0OtEJA6IbC9yR/+gp2vPutt8k0Z7qEta/zFPfVMXLNGk3m6UL3UoD6os375Umlj8mYPlgJXh9w6RH8ZHQCr32SMWRDLzJP15SIepeysk+CJw3lmPyn7cWwYUH8SY91V4Co4NqG8Zxes9RF65cJSszLeu5Ts+uh+zQcr5xQxjk4CWxFk7AUU+71Vt5eIvaU3GBCvRqDfTa7pnOWzZQVVq0WreFbL65xmrlkM4pLV5LxbxngjM6MF+ukMb1sEzherkDoMVvRbpGhOsdAMq9bUO5f/o1/1JQ55963b96X7l2T/tt7Z52K93QkaCkl0dIIUkp2KlCnVUGLhP4E3Z22QVi8hig/GZ6WYY+/qpYIHObvuXaUNUH/A/NP7dW31Qj9rZdoWsDDsq4gulFyV0641sDql4gSgG8ZbBvsGu36jkog5ZUZyiaV29GVAhhnjSRlPW7baah+fG2LQ5fdKopE4nqY9EEnHZEOT45G4ujs6/nJ6PxCIsLqvr6atgPkohJwSWwRaoviegqgaDShFTTX1K6nZv77WgkJ/f56uREjH4ZnlwNUf8Xl+Phxfj49CdXDI1OQj8NoFImuSjiOkHVO0dnn+E46pdOk/ChUNLj+5jN+yZmU/PmlTWTnilBxZnLZbWj+cYP8ZlGLUQB6D7v6tVx4sa6K214caSuLesEMRYXYd4nXRhhtu9d42aUizM214xsmc7cuRhdXp2MxfDi6MvxL6O+eJIdN88gl9Ij0vsE73vtYQIEoEGkK2wp7yyYmpwwEqrb0y04wu5F5BqIFXd2sV/6QdqMvT/02R1mN2tUXs7pDWb4zLKIijIPPC9IZ2DfWz2xdBrsCHfpyCAcbjfljuUDeddBlYO4NkVM5YutwEhbx/sy+DNeVMDf9D90t3aVP+Ir+56XfmZT+bZYzZRuj+0z0UHJdnBWhaitoEwYc8vUVKUi6e4wP027FbRlbe6isfuyMWSSXHuIeRHGqwHluWBKNwhd0rwDK+MlrqdxMgb0NLtlUx0RtnoTwnkWoWJ7IVtb6zgzNcda6wYVmRXSYjS2X16dnwOhXj6zrTpKbGPid0AqhZTNFLdBkr96JwHSHwRJNvhdKf/d3X8n/nVg2bn47WDffWeXqyEoIbtPJYHxb7BMQ/1jS/sICdVd7NyjT1M413vxCM2F1j8qXIciFCYjn6ahqlw1+W++zYtk//yUFezQ6qs1HDJY43TtKZ+SqsNPX61QfsPvqVYjMcBBI6wEwkXiSUnxPNwFz5N6CjHX7s7/AFmKDOo='))
assert hashlib.sha256(payload).hexdigest() == 'fcc0614c429ff020418ab129b30f26746a4f32fa5fddb3f44a2b460426654a70'
RUNNER.write_bytes(payload)

if not (ROOT / "manifest.json").exists():
    matches = list(Path("/kaggle/input").rglob("sempointer_qwen35_frozen.bin"))
    assert len(matches) == 1, "Attach exactly one frozen bundle Dataset."
    with matches[0].open("rb") as stream:
        checksum = hashlib.file_digest(stream, "sha256").hexdigest()
    assert checksum == "ea23e4556c9fbdf8c14669fdcc078a026bf66b07e0e71728a7f7e06dbf5f22a1"
    with tarfile.open(matches[0], "r:gz") as archive:
        archive.extractall(WORK, filter="data")

helpers = runpy.run_path(str(RUNNER))
OUTPUT = WORK / helpers["OUTPUT_NAME"]
manifest = helpers["verify_bundle"](ROOT)
print("Frozen inputs verified:", len(manifest["rows"]), "main questions")

previous_archive = WORK / 'sempointer_kaggle_results.tar.gz'
previous_backup = WORK / 'sempointer_kaggle_results_before_attention_fix.tar.gz'
if (not (OUTPUT/'config.json').exists() and (WORK/'kaggle_results/config.json').exists()
        and previous_archive.exists() and not previous_backup.exists()):
    shutil.copy2(previous_archive, previous_backup)
    print('Previous result archive preserved:', previous_backup)

# Use a pip-less venv: Kaggle's ensurepip bootstrap previously failed.
healthy = False
if PYTHON.exists():
    probe = subprocess.run([
        str(PYTHON), "-c", "import pip, torch; assert torch.cuda.device_count() == 2"
    ], capture_output=True, text=True)
    healthy = probe.returncode == 0
if not healthy:
    venv = WORK / "venv"
    if venv.exists() or venv.is_symlink():
        venv.rename(WORK / f"venv.previous-{time.time_ns()}")
    subprocess.run([
        sys.executable, "-m", "venv", "--without-pip",
        "--system-site-packages", str(venv)
    ], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "--python", str(PYTHON),
        "install", "--no-cache-dir", "pip"
    ], check=True)

constraints = WORK / "constraints.txt"
constraints.write_text(f"torch=={torch.__version__}\n")
subprocess.run([
    str(PYTHON), "-m", "pip", "install", "--no-cache-dir",
    "--only-binary=:all:", "--constraint", str(constraints),
    "transformers==5.2.0", "accelerate==1.10.1", "bitsandbytes==0.50.2",
    "numpy==2.2.6", "ijson==3.4.0", "pyarrow==21.0.0"
], check=True)

# Download into one local directory, avoiding a duplicate weight cache.
download = r"""
import hashlib, json
from pathlib import Path
from huggingface_hub import snapshot_download
work = Path('/kaggle/working/sempointer')
manifest = json.loads((work/'frozen_bundle/manifest.json').read_text())
model = work/'models'/manifest['model_revision']
snapshot_download(manifest['model_id'], revision=manifest['model_revision'],
    local_dir=model, token=False, max_workers=2,
    allow_patterns=['*.json','*.safetensors','*.txt','*.model','*.jinja'])
with (model/'tokenizer.json').open('rb') as stream:
    assert hashlib.file_digest(stream,'sha256').hexdigest() == manifest['tokenizer_file_sha256']
index = json.loads((model/'model.safetensors.index.json').read_text())
assert all((model/name).is_file() for name in set(index['weight_map'].values()))
print('Pinned checkpoint ready:', model, flush=True)
"""
result = subprocess.run([str(PYTHON), "-u", "-c", download], check=True)

if not (OUTPUT/"config.json").exists():
    archives = ([Path(RESTORE_ARCHIVE)] if RESTORE_ARCHIVE else
                list(Path('/kaggle/input').rglob('sempointer_kaggle_results.tar.gz')))
    assert len(archives) <= 1, "Set RESTORE_ARCHIVE to the saved run you want to resume."
    if archives:
        restored = helpers["restore_results_archive"](archives[0], WORK)
        print('Results available at:', restored)

print('Setup ready. Mode:', RUN_MODE, '| launcher allowance:', HOURS, 'hours')


In [ ]:
# A fresh interpreter uses the pinned environment and the frozen evaluator.
import signal

command = [str(PYTHON), '-u', str(RUNNER), '--mode', RUN_MODE, '--hours', str(HOURS)]
process = subprocess.Popen(command, start_new_session=True)
try:
    returncode = process.wait()
except KeyboardInterrupt:
    process.send_signal(signal.SIGINT)
    try:
        process.wait(timeout=120)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)
        process.wait()
    raise

print('Launcher exit code:', returncode)
archive = WORK / 'sempointer_kaggle_results.tar.gz'
if archive.exists():
    print('Download/save this result archive:', archive)
if returncode:
    for log in sorted(OUTPUT.glob('gpu*/worker.log')):
        print('\nLast worker log lines:', log)
        print('\n'.join(log.read_text(errors='replace').splitlines()[-35:]))
    raise RuntimeError('Evaluation paused with an error. Raw checkpoints were retained; inspect the logs above.')


In [ ]:
from IPython.display import FileLink, display

projection = OUTPUT / 'pilot_projection.json'
if projection.exists():
    report = json.loads(projection.read_text())
    print('Projected remaining wall hours:', round(report['projected_wall_seconds']/3600, 2))
    print(report['caution'])

archive = WORK / 'sempointer_kaggle_results.tar.gz'
if archive.exists():
    print('Results are also available in Kaggle’s Output panel under sempointer/.')
    display(FileLink(str(archive.relative_to(Path('/kaggle/working')))))
